# 문제 유형 이해와 scikit-learn estimator 선택

이 노트북은 세 가지 데이터셋을 이용해 classification, regression, clustering 문제에서 어떤 estimator를 선택하고 어떻게 `fit`, `predict`, `score`, `fit_predict`를 쓰는지 확인합니다.  
각 데이터셋을 학습하기 전에 **데이터 로딩 → 피처 설명 → 데이터 미리보기 → 시각화**를 먼저 수행하도록 구성했습니다.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn
from IPython.display import display

plt.rcParams["figure.figsize"] = (7, 4)
plt.rcParams["axes.grid"] = True

print("scikit-learn version:", sklearn.__version__)

# Iris classification

## 데이터 로딩 및 피처 설명

**목표**: `iris` 데이터의 피처, target class, 샘플 수를 먼저 확인한 뒤 classification estimator를 선택합니다.

**피처 설명**

- `sepal length (cm)`: 꽃받침 길이
- `sepal width (cm)`: 꽃받침 너비
- `petal length (cm)`: 꽃잎 길이
- `petal width (cm)`: 꽃잎 너비
- `target`: 붓꽃 품종 번호. `0=setosa`, `1=versicolor`, `2=virginica`

이 데이터는 정답 label이 있는 **지도학습 classification** 문제입니다.


In [ ]:
from sklearn.datasets import load_iris

In [ ]:
iris = load_iris(as_frame=True)
iris_df = iris.frame.copy()
iris_df["target_name"] = iris_df["target"].map(lambda x: iris.target_names[x])

In [ ]:
X = iris.data.to_numpy()
y = iris.target.to_numpy()

In [ ]:
print("full dataset:", X.shape, y.shape)
print("feature names:", iris.feature_names)
print("target names:", iris.target_names.tolist())
print("first labels:", y[:10].tolist(), np.unique(y))

In [ ]:
# 데이터 미리보기
display(iris_df.head())

In [ ]:
# class별 샘플 수
iris_df["target_name"].value_counts()

In [ ]:
# 기초 통계
iris_df[iris.feature_names].describe().T

## 데이터 시각화

학습 전에 class 균형과 피처 분포를 확인합니다. 특히 petal 관련 피처는 품종 구분에 도움이 되는 패턴이 잘 보입니다.

In [ ]:
# class별 샘플 수
fig, ax = plt.subplots()
iris_df["target_name"].value_counts().sort_index().plot(kind="bar", ax=ax)
ax.set_title("Iris class counts")
ax.set_xlabel("target class")
ax.set_ylabel("count")
plt.xticks(rotation=0)
plt.show()

In [ ]:
# petal length와 petal width의 관계
fig, ax = plt.subplots()
for name, group in iris_df.groupby("target_name"):
    ax.scatter(
        group["petal length (cm)"],
        group["petal width (cm)"],
        label=name,
        alpha=0.8,
    )
ax.set_title("Iris: petal length vs petal width")
ax.set_xlabel("petal length (cm)")
ax.set_ylabel("petal width (cm)")
ax.legend(title="species")
plt.show()

In [ ]:
# 전체 피처 분포
iris_df[iris.feature_names].hist(bins=15, figsize=(10, 6))
plt.suptitle("Iris feature distributions", y=1.02)
plt.tight_layout()
plt.show()

## 학습 : train/test split 후 LinearSVC

**목표**: 같은 데이터로 학습과 채점을 동시에 하지 않도록 holdout set을 만들고, 작은 tabular classification의 첫 후보로 `LinearSVC`를 실행합니다.


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import LinearSVC

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, stratify=y, random_state=42
)
print("train/test:", X_train.shape, X_test.shape)

In [ ]:
model = make_pipeline(
    StandardScaler(),
    LinearSVC(random_state=42)
)

In [ ]:
# 학습
model.fit(X_train, y_train)

## predict와 score 비교

**목표**: `predict`는 label을 반환하고, `score`는 test set의 accuracy를 반환한다는 차이를 확인합니다.

In [ ]:
pred = model.predict(X_test[:5])
score = model.score(X_test, y_test)

print("first predictions:", pred.tolist())
print("first label:", y_test[:5])
print("test accuracy:", round(score, 3))
print("prediction shape:", pred.shape)

# Diabetes regression

## Diabetes regression 데이터 로딩 및 피처 설명

**목표**: 숫자 `y`라도 class 번호가 아니라 연속값이면 regression estimator를 써야 함을 확인합니다.

**피처 설명**

- `age`: 나이
- `sex`: 성별 코드
- `bmi`: 체질량지수
- `bp`: 평균 혈압
- `s1` ~ `s6`: 혈청 검사 관련 수치
- `target`: 기준 시점으로부터 1년 뒤 질병 진행 정도를 나타내는 연속값

이 데이터는 target이 연속값인 **지도학습 regression** 문제입니다.


In [ ]:
from sklearn.datasets import load_diabetes

# scaled=False를 사용하면 원래 단위에 가까운 값으로 데이터를 살펴볼 수 있습니다.
diabetes = load_diabetes(scaled=False, as_frame=True)
diabetes_df = diabetes.frame.copy()

X_reg = diabetes.data.to_numpy()
y_reg = diabetes.target.to_numpy()

print("diabetes:", X_reg.shape, y_reg.shape)
print("feature names:", diabetes.feature_names)
print("first targets:", [round(v, 1) for v in y_reg[:5]])

print()
print("[데이터 미리보기]")
display(diabetes_df.head())

print()
print("[기초 통계]")
display(diabetes_df.describe().T)


## Diabetes 데이터 시각화

학습 전에 target 분포와 주요 피처와 target의 관계를 확인합니다. 회귀 문제에서는 피처와 연속 target 사이의 상관관계를 먼저 보는 것이 유용합니다.

In [ ]:
# target 분포
fig, ax = plt.subplots()
diabetes_df["target"].hist(bins=20, ax=ax)
ax.set_title("Diabetes target distribution")
ax.set_xlabel("target")
ax.set_ylabel("count")
plt.show()

# BMI와 target의 관계
fig, ax = plt.subplots()
ax.scatter(diabetes_df["bmi"], diabetes_df["target"], alpha=0.7)
ax.set_title("Diabetes: BMI vs target")
ax.set_xlabel("bmi")
ax.set_ylabel("target")
plt.show()

# 각 피처와 target의 상관계수
corr_with_target = diabetes_df.corr(numeric_only=True)["target"].drop("target").sort_values()
fig, ax = plt.subplots(figsize=(8, 5))
corr_with_target.plot(kind="barh", ax=ax)
ax.set_title("Correlation with diabetes target")
ax.set_xlabel("correlation")
plt.show()


## Ridge regression 학습 및 예측

**목표**: regression estimator인 `Ridge`를 학습하고, `score`가 R² 값을 반환한다는 점을 확인합니다.


In [ ]:
from sklearn.linear_model import Ridge

X_train_reg, X_test_reg, y_train_reg, y_test_reg = train_test_split(
    X_reg, y_reg, test_size=0.25, random_state=42
)

reg = Ridge(alpha=1.0)
reg.fit(X_train_reg, y_train_reg)

print("r2:", round(reg.score(X_test_reg, y_test_reg), 3))
print("first predictions:", [round(v, 1) for v in reg.predict(X_test_reg[:3])])
print("first label:", y_test_reg[:3])

**해석 + 다음 단계**: regression의 `predict`는 class label이 아니라 float 예측값을 반환합니다. 다음 단계에서는 `y` 없이 `fit_predict`를 쓰는 clustering 예제를 봅니다.


# blobs clustering

## `make_blobs` clustering 데이터 생성 및 피처 설명

**목표**: sample generator로 만든 cluster fixture에서 실제 학습에 들어가기 전에 synthetic 데이터의 구조를 먼저 확인합니다.

**피처 설명**

- `feature_0`: 2차원 평면의 첫 번째 좌표
- `feature_1`: 2차원 평면의 두 번째 좌표
- `true_cluster`: `make_blobs`가 검산용으로 만들어 준 정답 cluster 번호. 실제 KMeans 학습에는 사용하지 않습니다.

이 데이터는 clustering 구조를 관찰하기 위한 **비지도학습 예제**입니다. `true_cluster`는 평가용으로만 사용합니다.


In [ ]:
from sklearn.datasets import make_blobs

X_blob, y_true_blob = make_blobs(
    n_samples=300, centers=3, cluster_std=0.60, random_state=0
)

blob_df = pd.DataFrame(X_blob, columns=["feature_0", "feature_1"])
blob_df["true_cluster"] = y_true_blob

print("synthetic:", X_blob.shape)
print("first true labels:", y_true_blob[:10].tolist())

print()
print("[데이터 미리보기]")
display(blob_df.head())

print()
print("[cluster별 샘플 수: 검산용 label]")
display(blob_df["true_cluster"].value_counts().sort_index().rename("count"))

print()
print("[기초 통계]")
display(blob_df[["feature_0", "feature_1"]].describe().T)


## Blob 데이터 시각화

KMeans 학습 전에 데이터가 2차원 공간에서 어떻게 모여 있는지 확인합니다. 여기서는 설명을 위해 generator가 제공한 `true_cluster`를 색 구분에 사용하지만, 실제 clustering 학습에는 이 값을 넣지 않습니다.


In [ ]:
fig, ax = plt.subplots()
for cluster_id, group in blob_df.groupby("true_cluster"):
    ax.scatter(
        group["feature_0"],
        group["feature_1"],
        label=f"true cluster {cluster_id}",
        alpha=0.8,
    )
ax.set_title("Synthetic blobs before KMeans")
ax.set_xlabel("feature_0")
ax.set_ylabel("feature_1")
ax.legend()
plt.show()

## KMeans로 비지도 API 확인

**목표**: `KMeans.fit_predict`와 `KMeans.transform`의 출력을 확인합니다. `fit_predict`에는 `y_true_blob`를 넣지 않습니다.

In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score

km = KMeans(n_clusters=3, n_init=10, random_state=0)
labels = km.fit_predict(X_blob)
distances = km.transform(X_blob[:5])

print("cluster centers:", km.cluster_centers_.round(2).tolist())
print("first predict labels:", labels[:10].tolist())
print("distance shape:", distances.shape)
print("ARI for teaching check:", round(adjusted_rand_score(y_true_blob, labels), 3))


## KMeans 예측 cluster 시각화

아래 그래프는 KMeans가 예측한 cluster label로 같은 데이터를 다시 표시한 것입니다. 학습 전 데이터 구조와 학습 후 예측 label을 비교해 볼 수 있습니다.

In [ ]:
blob_df["kmeans_label"] = labels

fig, ax = plt.subplots()
for cluster_id, group in blob_df.groupby("kmeans_label"):
    ax.scatter(
        group["feature_0"],
        group["feature_1"],
        label=f"predicted cluster {cluster_id}",
        alpha=0.8,
    )

centers = pd.DataFrame(km.cluster_centers_, columns=["feature_0", "feature_1"])
ax.scatter(
    centers["feature_0"],
    centers["feature_1"],
    marker="X",
    s=200,
    label="centers",
)
ax.set_title("Synthetic blobs after KMeans")
ax.set_xlabel("feature_0")
ax.set_ylabel("feature_1")
ax.legend()
plt.show()


In [ ]:
distances

# 세 문제 유형을 한 표로 정리

**목표**: 방금 실행한 세 예제를 `y`의 의미, estimator, 주요 API 기준으로 정리합니다.

In [ ]:
records = [
    {
        "data": "iris",
        "task": "classification",
        "target": "category label",
        "estimator": "LinearSVC pipeline",
        "main_api": "fit / predict / score",
    },
    {
        "data": "diabetes",
        "task": "regression",
        "target": "continuous quantity",
        "estimator": "Ridge",
        "main_api": "fit / predict / score",
    },
    {
        "data": "blobs",
        "task": "clustering",
        "target": "no target for fitting",
        "estimator": "KMeans",
        "main_api": "fit_predict / transform",
    },
]

summary_df = pd.DataFrame(records)
display(summary_df)

for row in records:
    print(f"{row['data']}: {row['task']}")
    print(f"  target: {row['target']}")
    print(f"  estimator: {row['estimator']}")
    print(f"  api: {row['main_api']}")
